# Face Detection

Two methods:
1. **MediaPipe** (uses .tflite model) — if mediapipe is installed
2. **Haar Cascade** (built into OpenCV) — always works

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline

base = os.path.dirname(os.getcwd())

In [ ]:
img_path = os.path.join(base, 'IMG_7139.png')
if not os.path.exists(img_path):
    img_path = os.path.join(base, 'IMG_1626.jpg')
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f'Could not load image from {img_path}')
print(f'Loaded: {os.path.basename(img_path)} ({img.shape[1]}x{img.shape[0]})')

---
## Method 1: MediaPipe Face Detection

In [ ]:
try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

    model_path = os.path.join(base, 'face_detection_full_range.tflite')
    if not os.path.exists(model_path):
        model_path = os.path.join(base, 'face_detection_short_range.tflite')

    opts = vision.FaceDetectorOptions(
        base_options=python.BaseOptions(model_asset_path=model_path),
        running_mode=vision.RunningMode.IMAGE,
        min_detection_confidence=0.3
    )
    detector = vision.FaceDetector.create_from_options(opts)

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_img)

    img_mp = img.copy()
    count = 0
    for d in result.detections:
        b = d.bounding_box
        cv2.rectangle(img_mp,
                      (b.origin_x, b.origin_y),
                      (b.origin_x + b.width, b.origin_y + b.height),
                      (0, 255, 0), 2)
        count += 1

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img_mp, cv2.COLOR_BGR2RGB))
    plt.title(f'MediaPipe - Faces: {count}')
    plt.axis('off')
    plt.show()
    print(f'MediaPipe detected {count} face(s)')

except Exception as e:
    print(f'MediaPipe not available: {e}')
    print('Falling back to Haar Cascade...')

---
## Method 2: Haar Cascade Face Detection (fallback)

This uses OpenCV's built-in cascade classifier — no extra installs needed.

In [ ]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
if face_cascade.empty():
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt.xml')

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

img_hc = img.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(img_hc, (x, y), (x + w, y + h), (0, 255, 0), 3)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img_hc, cv2.COLOR_BGR2RGB))
plt.title(f'Haar Cascade - Faces Detected: {len(faces)}')
plt.axis('off')
plt.show()
print(f'Haar Cascade detected {len(faces)} face(s)')
if len(faces) == 0:
    print('Tip: Try lower scaleFactor (e.g. 1.05) or lower minNeighbors (e.g. 3)')